# Ex-ante Probabilistic LCA — structured workflow

A single **Python** pipeline (presample → Monte Carlo → Global Sensitivity Analysis) on a parameterised Brightway2 model. It consolidates a workflow that originally spanned three tools — **R** (presampling), **Activity Browser** (LCA), **MATLAB** (GSA) — into one notebook. Method, model & case study: Blanco et al. (2024), *J. Industrial Ecology* (see README / footer).

**We build up from simple to full:**

1. **Shared engine** — samplers + the Monte-Carlo loop (defined once, reused).
2. **Part A — simple case:** module performance, **4 parameters** (electricity functional unit).
3. **Part B — full model:** all **26 factors** (the thesis t=0 model).

**The one principle that makes it reproducible:** every parameterised exchange is overwritten from the parameters in every iteration, so the LCA score is a pure function of the sample matrix — immune to leftover database state.

> **Kernel:** use a **numpy < 2** environment (`ab_new`, `ab`, `premise`). `bw2data 3.6.x` crashes on numpy ≥ 2 (`np.NaN` removed). The check below warns you.

## 1. Imports & kernel check

In [ ]:
import ast
import numpy as np, pandas as pd, csv
import matplotlib.pyplot as plt
import seaborn as sns
from scipy.stats import norm, uniform, triang, lognorm, beta, bernoulli
import bw2data as bd, brightway2 as bw
from SALib.analyze import delta

if int(np.__version__.split('.')[0]) >= 2:
    print(f"WARNING: numpy {np.__version__}. bw2data 3.6.x needs numpy < 2 "
          f"(LCA crashes with 'np.NaN removed'). Switch to the ab_new / ab / premise kernel.")
else:
    print(f"numpy {np.__version__} OK")

## 2. Environment binding

The only Brightway2-specific identifiers — change these for a new project.

In [ ]:
PROJECT           = "FINAL"
PARAMETERISED_DBS = ["SiTaSol_F2v1a", "IEA_PVPS_2020"]   # every DB with formula exchanges
FU_DB, FU_CODE    = "SiTaSol_F2v1a", "ba8e2884bda6251e448284b8d4e9cc15_copy1"
METHOD            = ('ILCD 2.0 2018 midpoint', 'climate change', 'climate change total')

bd.projects.set_current(PROJECT)
fu     = bw.Database(FU_DB).get(FU_CODE)
method = bw.Method(METHOD)
print('Project:', bd.projects.current); print('FU     :', fu); print('Method :', method.name)

## 3. Shared engine — samplers + the Monte-Carlo loop

Defined once and reused by both cases. `run_mc` writes every `formula` exchange from the parameters each iteration and runs the LCA; the small helpers keep Parts A and B short.

In [ ]:
def pert(size, min, mode, max, shape=4):
    a = 1 + shape*(mode-min)/(max-min); b = 1 + shape*(max-mode)/(max-min)
    return beta.rvs(a, b, size=size)*(max-min)+min
def _bernoulli(size, p):                      # scalar p -> draw size; array p -> dependent draw
    return bernoulli.rvs(p, size=size) if np.ndim(p)==0 else bernoulli.rvs(p)
SAMPLERS = {
    'pert':       lambda size, **a: pert(size, **a),
    'normal':     lambda size, loc, scale: norm.rvs(loc, scale, size=size),
    'uniform':    lambda size, low, width: uniform.rvs(low, width, size=size),   # support [low, low+width]
    'triangular': lambda size, c, loc, scale: triang.rvs(c, loc, scale, size=size),
    'lognormal':  lambda size, s, scale: lognorm.rvs(s, scale=scale, size=size), # s=sigma of log; median=scale
    'beta':       lambda size, a, b: beta.rvs(a, b, size=size),
    'bernoulli':  _bernoulli,
}

def sample_registry(PARAMETERS, N, seed=42):
    """Draw all parameters in list order (a string arg = reuse an earlier sample)."""
    np.random.seed(seed); s = {}
    for p in PARAMETERS:
        a = {k:(s[v] if isinstance(v, str) else v) for k, v in p['args'].items()}
        s[p['name']] = SAMPLERS[p['dist']](size=N, **a)
    names = [p['name'] for p in PARAMETERS]
    return names, np.column_stack([s[n] for n in names]), s

def run_mc(X, names, fexc, fu, method, n_iter=None):
    """State-independent: all formula exchanges are overwritten each iteration."""
    n_iter = n_iter or X.shape[0]; scores = np.empty(n_iter)
    for i in range(n_iter):
        p = dict(zip(names, X[i].tolist()))
        for e in fexc:
            try: e['amount'] = float(eval(e['formula'], {'__builtins__': None}, p)); e.save()
            except Exception: pass          # formula needs a param not in this case -> keep default
        lca = bw.LCA({fu: 1}, method.name); lca.lci(); lca.lcia(); scores[i] = lca.score
    return scores

def scatter_grid(names, X, Y, ncol=6, title='Model output vs each input', mark=()):
    nrow = int(np.ceil(len(names)/ncol))
    fig, axes = plt.subplots(nrow, ncol, figsize=(ncol*2.5, max(1, nrow)*2.1), sharey=True)
    axes = np.atleast_1d(axes).flatten()
    for j, nm in enumerate(names):
        axes[j].scatter(X[:, j], Y, s=3, alpha=0.12, color='#4c72b0')
        axes[j].set_xlabel(('* '+nm) if nm in mark else nm, fontsize=8); axes[j].tick_params(labelsize=6)
    for k in range(len(names), len(axes)): axes[k].axis('off')
    fig.suptitle(title); plt.tight_layout(rect=[0, 0, 1, 0.96]); plt.show()

def gsa_table(names, X, Y):
    prob = {'num_vars': len(names), 'names': names, 'bounds': list(zip(X.min(0), X.max(0)))}
    return delta.analyze(prob, X, Y).to_df()

# All formula exchanges across every parameterised database (used by both cases):
formula_exchanges = [e for dbn in PARAMETERISED_DBS for a in bw.Database(dbn) for e in a.exchanges() if 'formula' in e]
print('formula exchanges:', len(formula_exchanges))

# Part A — Simple case: module performance (4 parameters)

Start here. Functional unit = **1 kWh of electricity**, varying only the four generation parameters `Eff_PV`, `PR_PV`, `LT`, `Irrad`. Only the electricity-production exchange varies, via `1/(Eff_PV·PR_PV·LT·Irrad)/21.429`; all other exchanges keep their database value.

Because the score is **purely multiplicative** in these four, the δ **ranking is reproducible regardless of background state** — a clean first example. (Run on a freshly restored project if you also want meaningful absolute numbers.)

In [ ]:
ELEC_PARAMS = [
    dict(name='Eff_PV', dist='pert',    args=dict(min=0.25, mode=0.28, max=0.31, shape=4)),
    dict(name='PR_PV',  dist='pert',    args=dict(min=0.8,  mode=0.85, max=0.9,  shape=4)),
    dict(name='LT',     dist='normal',  args=dict(loc=30, scale=5)),
    dict(name='Irrad',  dist='uniform', args=dict(low=1500, width=500)),
]
names4, X4, _ = sample_registry(ELEC_PARAMS, 5000)
Y4 = run_mc(X4, names4, formula_exchanges, fu, method)   # only the electricity exchange updates
print(f'mean = {Y4.mean():.4f} kg CO2 eq/kWh   (P5={np.percentile(Y4,5):.4f}, P95={np.percentile(Y4,95):.4f})')

### A — visualise output and screen each input

In [ ]:
fig, ax = plt.subplots(1, 2, figsize=(10, 3.4))
ax[0].hist(Y4, bins=45, color='#4c72b0'); ax[0].set_title('MC output'); ax[0].set_xlabel('kg CO₂ eq/kWh')
ax[1].boxplot(Y4); ax[1].set_xticks([]); ax[1].set_title('Boxplot')
plt.tight_layout(); plt.show()
scatter_grid(names4, X4, Y4, ncol=4, title='Output vs each input (4-parameter case)')

### A — global sensitivity (Borgonovo δ)

In [ ]:
df4 = gsa_table(names4, X4, Y4).sort_values('delta', ascending=False)
print(df4[['delta', 'S1']].round(4).to_string())
plt.figure(figsize=(6, 2.2)); o = df4.sort_values('delta')
plt.barh(range(len(o)), o['delta'], color='#c44e52'); plt.yticks(range(len(o)), o.index)
plt.xlabel('Borgonovo δ'); plt.title('4-parameter case — sensitivity'); plt.tight_layout(); plt.show()
# Expected: LT >> Irrad > Eff_PV > PR_PV

# Part B — Full model (26 factors)

Now the complete t=0 model: all 21 direct parameters **plus** the five `pi_*` success-probabilities (the binary design choices' chances). The `pi_*` carry no LCA formula; they enter the GSA indirectly via the `bin_*` switches they parameterise.

## B1. Parameter registry  ← edit here

Single source of truth: one row per parameter. `name` must match the formula variable (except the indirect `pi_*`). A string `args` value (`p="pi_FM"`) reuses an earlier sample. `group=None` drops a factor entirely; `indirect=True` marks the `pi_*`.

In [ ]:
_S = np.log(1.22)   # geometric SD for all lognormal inventory parameters
PARAMETERS = [
    dict(name='Eff_PV',        dist='pert',       args=dict(min=0.25, mode=0.28, max=0.31, shape=4), group='project',  desc='Panel efficiency'),
    dict(name='PR_PV',         dist='pert',       args=dict(min=0.8,  mode=0.85, max=0.9,  shape=4), group='project',  desc='Performance ratio'),
    dict(name='LT',            dist='normal',     args=dict(loc=30, scale=5),                        group='project',  desc='Panel lifetime'),
    dict(name='Irrad',         dist='uniform',    args=dict(low=1500, width=500),                    group='project',  desc='Irradiation'),
    dict(name='RT_movpe',      dist='pert',       args=dict(min=0.5, mode=3.5, max=3.5, shape=4),    group='project',  desc='MOVPE runtime'),
    dict(name='P_movpe_tool',  dist='pert',       args=dict(min=1, mode=509, max=509),               group='project',  desc='MOVPE tool power'),
    dict(name='Zeol_scrub',    dist='triangular', args=dict(c=0, loc=2.55, scale=7.65-2.55),         group='project',  desc='Scrubber granulate'),
    dict(name='Cu_zeol',       dist='pert',       args=dict(min=0.2, mode=0.3, max=0.7, shape=4),    group='project',  desc='Granulate Cu fraction'),
    dict(name='Cu_rec',        dist='bernoulli',  args=dict(p=0.5),                                  group='project',  desc='Cu recycling switch'),
    dict(name='pi_NPsynthCu',  dist='pert',       args=dict(min=0.5, mode=0.7, max=0.8),             group='project',  indirect=True, desc='P(chem synth Cu)'),
    dict(name='bin_NPsynthCu', dist='bernoulli',  args=dict(p='pi_NPsynthCu'),                       group='project',  desc='Cu synthesis choice'),
    dict(name='pi_NPsynthAg',  dist='pert',       args=dict(min=0.5, mode=0.7, max=0.8),             group='project',  indirect=True, desc='P(chem synth Ag)'),
    dict(name='bin_NPsynthAg', dist='bernoulli',  args=dict(p='pi_NPsynthAg'),                       group='project',  desc='Ag synthesis choice'),
    dict(name='pi_CuSint',     dist='pert',       args=dict(min=0.1, mode=0.2, max=0.3, shape=4),    group='project',  indirect=True, desc='P(laser sinter Cu)'),
    dict(name='bin_CuSint',    dist='bernoulli',  args=dict(p='pi_CuSint'),                          group='project',  desc='Cu sintering choice'),
    dict(name='pi_AgSint',     dist='uniform',    args=dict(low=0, width=1),                         group='project',  indirect=True, desc='P(laser sinter Ag)'),
    dict(name='bin_AgSint',    dist='bernoulli',  args=dict(p='pi_AgSint'),                          group='project',  desc='Ag sintering choice'),
    dict(name='pi_FM',         dist='beta',       args=dict(a=4, b=2),                               group='project',  indirect=True, desc='P(Cu vs Ag nanoink)'),
    dict(name='bin_FM',        dist='bernoulli',  args=dict(p='pi_FM'),                              group='project',  desc='Front-metal choice'),
    dict(name='Elec_Siem',     dist='lognormal',  args=dict(s=_S, scale=110),                        group='activity', desc='Siemens electricity'),
    dict(name='Heat_Siem',     dist='lognormal',  args=dict(s=_S, scale=185),                        group='activity', desc='Siemens heat'),
    dict(name='Elec_CZ',       dist='lognormal',  args=dict(s=_S, scale=85.26),                      group='activity', desc='Czochralski electricity'),
    dict(name='scSi_CZ',       dist='lognormal',  args=dict(s=_S, scale=1.07),                       group='activity', desc='Czochralski silicon'),
    dict(name='Al_panel',      dist='lognormal',  args=dict(s=_S, scale=2.63),                       group='project',  desc='Aluminium in panel'),
    dict(name='Glass_panel',   dist='lognormal',  args=dict(s=_S, scale=10.08),                      group='project',  desc='Glass in panel'),
    dict(name='Elec_panel',    dist='lognormal',  args=dict(s=_S, scale=4.71),                       group='project',  desc='Panel electricity'),
]
MODEL_PARAMS = [p['name'] for p in PARAMETERS if p['group'] is not None]
INDIRECT     = {p['name'] for p in PARAMETERS if p.get('indirect')}
print(f'{len(MODEL_PARAMS)} model parameters ({len(INDIRECT)} indirect: {sorted(INDIRECT)})')

## B2. Presampling → CSV

In [ ]:
_, _, sampled = sample_registry(PARAMETERS, 10000)
var_level = {p['name']: p['group'] for p in PARAMETERS if p['group'] is not None}
df_T = pd.DataFrame({n: sampled[n] for n in MODEL_PARAMS}).T
df_T.index.name = 'Name'; df_T.insert(0, 'Group', df_T.index.map(var_level))
df_T.to_csv('PROB_X_t0.csv'); print('Saved PROB_X_t0.csv', df_T.shape)

## B3. Load X

In [ ]:
CSV_FILE = 'PROB_X_t0.csv'
with open(CSV_FILE) as f:
    r = csv.reader(f); next(r); rows = list(r)
param_names = [x[0] for x in rows]
X = np.array([x[2:] for x in rows], dtype=float).T
print('X shape:', X.shape)

## B4. Validation gate + introspection

Cross-check the parameters against the model formulas — catches the classic 'forgot a database' bug. Indirect `pi_*` are recognised, not flagged.

In [ ]:
declared = set(param_names)
used = set()
for e in formula_exchanges:
    used |= {n.id for n in ast.walk(ast.parse(e['formula'], mode='eval')) if isinstance(n, ast.Name)}
unresolved = used - declared
indirect   = (declared & INDIRECT) - used
unused     = declared - used - INDIRECT
if unresolved: raise ValueError(f'Formulas need params not in the CSV: {sorted(unresolved)}')
if indirect: print(f'ℹ indirect factors (act via their bin_* switch): {sorted(indirect)}')
print('⚠ unused:', sorted(unused)) if unused else print('✓ every direct parameter is used by a formula')

In [ ]:
rows_ = []
for e in formula_exchanges:
    vs = sorted({n.id for n in ast.walk(ast.parse(e['formula'], mode='eval')) if isinstance(n, ast.Name)})
    rows_.append({'to activity': e.output['name'][:38], 'input': e.input['name'][:30],
                  'formula': e['formula'], 'uses': ', '.join(vs)})
pd.DataFrame(rows_)

## B5. Verification (5 scenarios vs reference)

Reproducible clean-Python reference (= the original parameter approach):
```
0.136787  0.188661  0.105696  0.148844  0.156320
```

In [ ]:
ref = [0.136787, 0.188661, 0.105696, 0.148844, 0.156320]
Yt = run_mc(X, param_names, formula_exchanges, fu, method, n_iter=5)
print(f"{'Sc':>3}  {'Python':>12}  {'reference':>12}  {'Diff':>12}")
print('-'*46)
for i,(p,r) in enumerate(zip(Yt, ref)): print(f'{i:>3}  {p:>12.6f}  {r:>12.6f}  {p-r:>+12.6f}')

## B6. Full MC run (10 000 iterations)

Run after the verification matches.

In [ ]:
import time; t0 = time.time()
Y = run_mc(X, param_names, formula_exchanges, fu, method)
print(f'Done in {time.time()-t0:.0f}s.  mean={Y.mean():.4f}  std={Y.std():.4f}')
pd.DataFrame({'score': Y}).to_csv('Model_results_python.csv', index=False)

## B7. Visualise — distribution + scatter screening

In [ ]:
# Y = pd.read_csv('Model_results_python.csv')['score'].values   # load if needed
fig, ax = plt.subplots(1, 2, figsize=(10, 3.6))
ax[0].hist(Y, bins=50, color='#4c72b0'); ax[0].set_title('Distribution of GWI'); ax[0].set_xlabel('kg CO₂ eq/kWh')
ax[1].boxplot(Y); ax[1].set_xticks([]); ax[1].set_title('Boxplot')
plt.tight_layout(); plt.show()
scatter_grid(param_names, X, Y, ncol=6, title='Model output vs each input (26 factors)', mark=INDIRECT)

## B8. GSA — Borgonovo δ

In [ ]:
df_gsa = gsa_table(param_names, X, Y)
print(df_gsa.sort_values('delta', ascending=False)[['delta','S1']].round(4).to_string())

In [ ]:
order = df_gsa.sort_values('delta', ascending=False).index.tolist()
labels = [('* '+n if n in INDIRECT else n) for n in order]
plt.figure(figsize=(0.62*len(order)+2, 2.9))
ax = sns.heatmap(df_gsa.loc[order, ['delta']].T, cmap='coolwarm', annot=True, fmt='.3f',
                 annot_kws={'size': 6}, cbar_kws={'label': 'Borgonovo δ'}, xticklabels=labels)
ax.set_yticks([]); plt.title('Global sensitivity (Borgonovo δ), sorted  (* = indirect pₓ)')
plt.tight_layout(); plt.savefig('GSA_heatmap.png', dpi=150); plt.show()
df_gsa.to_csv('GSA_results.csv'); print('Saved GSA_results.csv and GSA_heatmap.png')

---
## Provenance & how to cite

**Method & case study.** III‑V/Si tandem PV case study and the Safe‑and‑Sustainable‑by‑Design framework: **Carlos Felipe Blanco** and colleagues (CML, Leiden University). The original analysis ran across **R** (presampling), the **Activity Browser** (LCA), and **MATLAB** (GSA). This notebook consolidates that into one reproducible Python pipeline — the contribution here is the consolidation.

**Cite:** Blanco, C. F., Behrens, P., Vijver, M. G., Peijnenburg, W. J. G. M., Quik, J. T. K., & Cucurachi, S. (2024). *A framework for guiding safe and sustainable‑by‑design innovation.* Journal of Industrial Ecology, 29(1). https://doi.org/10.1111/jiec.13609

New to the workflow? See `examples/` for two self‑contained teaching notebooks. Full background and the generalisation guide: `README.md` and `docs/WORKFLOW.md`.